# SNOM Raster Scan
2-D tip scan with per-point optical and mechanical amplitude + phase readout for harmonics 0–5.

## 0 — Configuration

In [11]:
HOST        = 'nea-server'
PATH_TO_DLL = ''
SPEED_UM_S  = 0.2          # tip travel speed µm/s

# Scan grid (all in µm)
X_START_UM = 51.35
Y_START_UM = 32.67
X_STEP_UM  = 0.1           # 100 nm pitch
Y_STEP_UM  = 0.05
NX         = 1
NY         = 500
SNAKE      = True          # serpentine scan order

# Sampling
T_INTEG_S  = 0.05       # integration time per point (s)
N_AVG      = 1             # number of integration windows averaged per point

## 1 — Imports

In [12]:
import asyncio
import time
from time import sleep

import nest_asyncio
import numpy as np
import matplotlib.pyplot as plt
import nea_tools

print('nea_tools version:', getattr(nea_tools, '__version__', 'unknown'))

nea_tools version: unknown


## 2 — Connect to nea-server

In [13]:
loop = asyncio.get_event_loop()
nest_asyncio.apply(loop)

print(f'Connecting to {HOST} ...')
loop.run_until_complete(nea_tools.connect(HOST, fingerprint=None, path_to_dll=PATH_TO_DLL))
print('Connected.')

Connecting to nea-server ...
Trying to download from nea-server
15:32:51.560 Nea.Client.Database Connecting to: Username=neaspec;Database=neaspec;Host=nea-server;Port=5432 
15:32:51.650 Nea.Client.SDK Waiting for initialization... 
15:32:51.664 Nea.Client.Hardware.Microscope Failed to resolve 10.227.127.150 - No such host is known. System.Net.Sockets.SocketException (0x00002AF9): No such host is known.
   at System.Net.Dns.GetHostEntryOrAddressesCore(IPAddress address, Boolean justAddresses, AddressFamily addressFamily, Nullable`1 startingTimestamp)
   at System.Net.Dns.<>c.<GetHostEntryOrAddressesCoreAsync>b__33_1(Object s, Int64 startingTimestamp)
   at System.Net.Dns.<>c__DisplayClass39_0`1.<RunAsync>b__0(Task <p0>, Object <p1>)
   at System.Threading.Tasks.ContinuationResultTaskFromTask`1.InnerInvoke()
   at System.Threading.ExecutionContext.RunFromThreadPoolDispatchLoop(Thread threadPoolThread, ExecutionContext executionContext, ContextCallback callback, Object state)
--- End of s

## 3 — Post-connect imports (injected by nea_tools)

In [14]:
from neaspec import context
from nea_tools.microscope import stream
import Nea.Client.SharedDefinitions as nea

print('context:', context)
print('nea    :', nea)

context: Nea.Client.SDK.ApplicationContext
nea    : <module 'Nea.Client.SharedDefinitions'>


## 4 — Move-tip helper

In [15]:
do_wait = False

def on_tip_position_moved(sender, args):
    print(f'Arrived at {args.Point}')
    global do_wait
    do_wait = False

def on_tip_position_moving(sender, args):
    print(f'Moving to {args.Point}')
    global do_wait
    do_wait = True

context.Logic.TipPositionMoved  += on_tip_position_moved
context.Logic.TipPositionMoving += on_tip_position_moving

def move_tip(x_um: float, y_um: float, speed_um_s: float = SPEED_UM_S) -> None:
    """Move tip to (x_um, y_um) and block until done."""
    move_args = nea.MoveTipPositionArgs(
        nea.Geometry.Point2D(x_um, y_um),
        speed_um_s / 1000.0,
    )
    context.Logic.MoveTipPosition.Execute(move_args)
    sleep(0.1)
    while do_wait:
        sleep(0.1)
    sleep(0.2)

print('move_tip() defined.')

move_tip() defined.


## 5 — Per-point sampler
Polls amplitudes for `T_INTEG_S` seconds and averages, then opens a `Stream()` for phase tail. Repeats `N_AVG` times and returns the mean.

In [ ]:
def sample_point(t_integ_s: float, n_avg: int):
    """Return (o_amp[6], o_phase[6], m_amp[6], m_phase[6]) averaged over n_avg windows."""
    o_amp   = np.zeros(6)
    m_amp   = np.zeros(6)
    o_phase = np.zeros(6)
    m_phase = np.zeros(6)

    for _ in range(n_avg):
        oa_acc = np.zeros(6)
        ma_acc = np.zeros(6)
        op_acc = np.zeros(6)
        mp_acc = np.zeros(6)
        count  = 0

        with stream.Stream() as s:
            t_end = time.time() + t_integ_s
            while time.time() < t_end:
                for h in range(6):
                    try: oa_acc[h] += s.data[f'O{h}A'][-1]
                    except Exception: pass
                    try: op_acc[h] += s.data[f'O{h}P'][-1]
                    except Exception: pass
                    try: ma_acc[h] += s.data[f'M{h}A'][-1]
                    except Exception: pass
                    try: mp_acc[h] += s.data[f'M{h}P'][-1]
                    except Exception: pass
                count += 1
                time.sleep(0.02)  # match ~20 ms stream cadence

        if count:
            o_amp   += oa_acc / count
            o_phase += op_acc / count
            m_amp   += ma_acc / count
            m_phase += mp_acc / count

    return o_amp / n_avg, o_phase / n_avg, m_amp / n_avg, m_phase / n_avg

print('sample_point() defined.')

## 6 — Allocate maps and define grid iterator

In [17]:
o_amp_map   = np.full((6, NY, NX), np.nan)
o_phase_map = np.full((6, NY, NX), np.nan)
m_amp_map   = np.full((6, NY, NX), np.nan)
m_phase_map = np.full((6, NY, NX), np.nan)

def grid_points():
    for iy in range(NY):
        xs = range(NX - 1, -1, -1) if (SNAKE and iy % 2 == 1) else range(NX)
        for ix in xs:
            yield ix, iy, X_START_UM + ix * X_STEP_UM, Y_START_UM + iy * Y_STEP_UM

print(f'Grid: {NX}x{NY} = {NX*NY} points  |  '
      f'step ({X_STEP_UM*1000:.0f}, {Y_STEP_UM*1000:.0f}) nm  |  '
      f'integ {T_INTEG_S}s x{N_AVG}')

Grid: 100x50 = 5000 points  |  step (100, 400) nm  |  integ 0.05s x1


## 7 — Scan

In [ ]:
total = NX * NY
t0    = time.time()

for k, (ix, iy, x_um, y_um) in enumerate(grid_points(), 1):
    move_tip(x_um, y_um)
    oa, op, ma, mp = sample_point(T_INTEG_S, N_AVG)
    o_amp_map  [:, iy, ix] = oa
    o_phase_map[:, iy, ix] = op
    m_amp_map  [:, iy, ix] = ma
    m_phase_map[:, iy, ix] = mp

    if k % max(total // 20, 1) == 0 or k == total:
        elapsed = time.time() - t0
        eta     = elapsed / k * (total - k)
        print(f'  {k:>4}/{total}  ({100*k/total:.0f}%)  '
              f'elapsed {elapsed:.0f}s  ETA {eta:.0f}s')

print('\nScan complete.')

Moving to 51.35,32.67
15:33:04.957 Nea.Client.Logic Moving tip to X=51.35µm, Y=32.67µm with 0.200µm/s 
Arrived at 51.35,32.67
Moving to 51.45,32.67
15:33:05.334 Nea.Client.Logic Moving tip to X=51.45µm, Y=32.67µm with 0.200µm/s 
Arrived at 51.45,32.67
Moving to 51.550000000000004,32.67
15:33:05.799 Nea.Client.Logic Moving tip to X=51.55µm, Y=32.67µm with 0.200µm/s 
Arrived at 51.550000000000004,32.67
Moving to 51.65,32.67
15:33:06.397 Nea.Client.Logic Moving tip to X=51.65µm, Y=32.67µm with 0.200µm/s 
Arrived at 51.65,32.67
Moving to 51.75,32.67
15:33:06.898 Nea.Client.Logic Moving tip to X=51.75µm, Y=32.67µm with 0.200µm/s 
Arrived at 51.75,32.67
Moving to 51.85,32.67
15:33:07.401 Nea.Client.Logic Moving tip to X=51.85µm, Y=32.67µm with 0.200µm/s 
Arrived at 51.85,32.67
Moving to 51.95,32.67
15:33:07.986 Nea.Client.Logic Moving tip to X=51.95µm, Y=32.67µm with 0.200µm/s 
Arrived at 51.95,32.67
Moving to 52.050000000000004,32.67
15:33:08.568 Nea.Client.Logic Moving tip to X=52.05µm, Y=

Arrived at 59.150000000000006,33.07
Moving to 59.050000000000004,33.07
15:34:10.197 Nea.Client.Logic Moving tip to X=59.05µm, Y=33.07µm with 0.200µm/s 
Arrived at 59.050000000000004,33.07
Moving to 58.95,33.07
15:34:10.672 Nea.Client.Logic Moving tip to X=58.95µm, Y=33.07µm with 0.200µm/s 
Arrived at 58.95,33.07
Moving to 58.85,33.07
15:34:11.185 Nea.Client.Logic Moving tip to X=58.85µm, Y=33.07µm with 0.200µm/s 
Arrived at 58.85,33.07
Moving to 58.75,33.07
15:34:11.790 Nea.Client.Logic Moving tip to X=58.75µm, Y=33.07µm with 0.200µm/s 
Arrived at 58.75,33.07
Moving to 58.650000000000006,33.07
15:34:12.368 Nea.Client.Logic Moving tip to X=58.65µm, Y=33.07µm with 0.200µm/s 
Arrived at 58.650000000000006,33.07
Moving to 58.550000000000004,33.07
15:34:12.984 Nea.Client.Logic Moving tip to X=58.55µm, Y=33.07µm with 0.200µm/s 
Arrived at 58.550000000000004,33.07
Moving to 58.45,33.07
15:34:13.449 Nea.Client.Logic Moving tip to X=58.45µm, Y=33.07µm with 0.200µm/s 
Arrived at 58.45,33.07
Movi

Moving to 52.95,33.07
15:34:43.508 Nea.Client.Logic Moving tip to X=52.95µm, Y=33.07µm with 0.200µm/s 
Arrived at 52.95,33.07
Moving to 52.85,33.07
15:34:44.129 Nea.Client.Logic Moving tip to X=52.85µm, Y=33.07µm with 0.200µm/s 
Arrived at 52.85,33.07
Moving to 52.75,33.07
15:34:44.718 Nea.Client.Logic Moving tip to X=52.75µm, Y=33.07µm with 0.200µm/s 
Arrived at 52.75,33.07
Moving to 52.65,33.07
15:34:45.341 Nea.Client.Logic Moving tip to X=52.65µm, Y=33.07µm with 0.200µm/s 
Arrived at 52.65,33.07
Moving to 52.550000000000004,33.07
15:34:45.907 Nea.Client.Logic Moving tip to X=52.55µm, Y=33.07µm with 0.200µm/s 
Arrived at 52.550000000000004,33.07
Moving to 52.45,33.07
15:34:46.377 Nea.Client.Logic Moving tip to X=52.45µm, Y=33.07µm with 0.200µm/s 
Arrived at 52.45,33.07
Moving to 52.35,33.07
15:34:47.009 Nea.Client.Logic Moving tip to X=52.35µm, Y=33.07µm with 0.200µm/s 
Arrived at 52.35,33.07
Moving to 52.25,33.07
15:34:47.486 Nea.Client.Logic Moving tip to X=52.25µm, Y=33.07µm with 

Arrived at 55.85,33.47
Moving to 55.95,33.47
15:35:17.221 Nea.Client.Logic Moving tip to X=55.95µm, Y=33.47µm with 0.200µm/s 
Arrived at 55.95,33.47
Moving to 56.050000000000004,33.47
15:35:17.823 Nea.Client.Logic Moving tip to X=56.05µm, Y=33.47µm with 0.200µm/s 
Arrived at 56.050000000000004,33.47
Moving to 56.150000000000006,33.47
15:35:18.431 Nea.Client.Logic Moving tip to X=56.15µm, Y=33.47µm with 0.200µm/s 
Arrived at 56.150000000000006,33.47
Moving to 56.25,33.47
15:35:18.987 Nea.Client.Logic Moving tip to X=56.25µm, Y=33.47µm with 0.200µm/s 
Arrived at 56.25,33.47
   250/5000  (5%)  elapsed 135s  ETA 2557s
Moving to 56.35,33.47
15:35:19.553 Nea.Client.Logic Moving tip to X=56.35µm, Y=33.47µm with 0.200µm/s 
Arrived at 56.35,33.47
Moving to 56.45,33.47
15:35:20.012 Nea.Client.Logic Moving tip to X=56.45µm, Y=33.47µm with 0.200µm/s 
Arrived at 56.45,33.47
Moving to 56.550000000000004,33.47
15:35:20.576 Nea.Client.Logic Moving tip to X=56.55µm, Y=33.47µm with 0.200µm/s 
Arrived at

Arrived at 60.75,33.870000000000005
Moving to 60.650000000000006,33.870000000000005
15:35:49.996 Nea.Client.Logic Moving tip to X=60.65µm, Y=33.87µm with 0.200µm/s 
Arrived at 60.650000000000006,33.870000000000005
Moving to 60.550000000000004,33.870000000000005
15:35:50.597 Nea.Client.Logic Moving tip to X=60.55µm, Y=33.87µm with 0.200µm/s 
Arrived at 60.550000000000004,33.870000000000005
Moving to 60.45,33.870000000000005
15:35:51.061 Nea.Client.Logic Moving tip to X=60.45µm, Y=33.87µm with 0.200µm/s 
Arrived at 60.45,33.870000000000005
Moving to 60.35,33.870000000000005
15:35:51.575 Nea.Client.Logic Moving tip to X=60.35µm, Y=33.87µm with 0.200µm/s 
Arrived at 60.35,33.870000000000005
Moving to 60.25,33.870000000000005
15:35:52.046 Nea.Client.Logic Moving tip to X=60.25µm, Y=33.87µm with 0.200µm/s 
Arrived at 60.25,33.870000000000005
Moving to 60.150000000000006,33.870000000000005
15:35:52.633 Nea.Client.Logic Moving tip to X=60.15µm, Y=33.87µm with 0.200µm/s 
Arrived at 60.150000000

## 8 — Plot maps

In [ ]:
%matplotlib widget

def plot_maps(maps, title, cmap='viridis', units=''):
    extent = [
        X_START_UM, X_START_UM + NX * X_STEP_UM,
        Y_START_UM, Y_START_UM + NY * Y_STEP_UM,
    ]
    fig, axes = plt.subplots(2, 3, figsize=(13, 8))
    for h, ax in enumerate(axes.flat):
        im = ax.imshow(
            maps[h], origin='lower', extent=extent,
            cmap=cmap, aspect='equal',
        )
        ax.set_title(f'h = {h}')
        ax.set_xlabel('X (µm)')
        ax.set_ylabel('Y (µm)')
        fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    fig.suptitle(title + (f'  [{units}]' if units else ''), fontsize=14)
    fig.tight_layout()
    plt.show()

plot_maps(o_amp_map,   'Optical amplitude  O0–O5')
plot_maps(o_phase_map, 'Optical phase  O0–O5',      cmap='twilight', units='rad')
plot_maps(m_amp_map,   'Mechanical amplitude  M0–M5')
plot_maps(m_phase_map, 'Mechanical phase  M0–M5',   cmap='twilight', units='rad')

## 9 — Disconnect

In [10]:
context.Logic.TipPositionMoved  -= on_tip_position_moved
context.Logic.TipPositionMoving -= on_tip_position_moving

try:
    nea_tools.disconnect()
    print('Disconnected.')
except Exception as e:
    print(f'Disconnect error (ignored): {e}')

15:31:10.280 Nea.Client.SDK Disposing... 
15:31:10.312 Nea.Client.RpcPreview[80000001] RpcPreview ignores dispose and can be reused 
15:31:10.312 Nea.Client.Hardware.Rpc Send thread (Nea.Client.Hardware.Rpc) stopped (disposed) 
15:31:10.312 Nea.Client.Hardware.Rpc Receive thread (Nea.Client.Hardware.Rpc) stopped (disposed) 
Disconnected.
